In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, welch, find_peaks
from scipy.stats import skew, kurtosis
from scipy.interpolate import interp1d
from scipy.signal import resample
from sklearn.metrics import mean_absolute_error


In [2]:
# Load from previous notebook’s saved files (you should save X and y first)
X = np.load(r"C:\Users\admin\OneDrive\Desktop\healthcare_ppg\data\processed\X_windows.npy")
y = np.load(r"C:\Users\admin\OneDrive\Desktop\healthcare_ppg\data\processed\y_labels.npy")

print("X shape:", X.shape)
print("y shape:", y.shape)


X shape: (1638, 3750)
y shape: (1638,)


In [3]:
def bandpass_filter(signal, lowcut, highcut, fs, order=4):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype='band')
    return filtfilt(b, a, signal)


In [4]:
def detect_r_peaks(signal, fs=125):
    peaks, _ = find_peaks(signal, distance=fs*0.4)  # ~150 bpm max
    return peaks


In [5]:
def build_amplitude_edr(signal, fs=125):
    peaks = detect_r_peaks(signal, fs)
    if len(peaks) < 5:
        return None, None
    amplitudes = signal[peaks]
    return peaks, amplitudes


In [6]:
def build_rr_interval_edr(signal, fs=125):
    peaks = detect_r_peaks(signal, fs)
    if len(peaks) < 5:
        return None, None
    rr_intervals = np.diff(peaks) / fs
    rr_times = peaks[:-1] / fs
    return rr_times, rr_intervals


In [7]:
def interpolate_signal(x, y, target_len):
    f = interp1d(x, y, kind='cubic', fill_value='extrapolate')
    x_uniform = np.linspace(x[0], x[-1], target_len)
    return f(x_uniform)

def downsample_signal(signal, original_fs=125, target_fs=10):
    duration = len(signal) / original_fs
    num_samples = int(duration * target_fs)
    return resample(signal, num_samples)


In [8]:
def extract_time_features(signal):
    return {
        'mean': np.mean(signal),
        'std': np.std(signal),
        'skew': skew(signal),
        'kurtosis': kurtosis(signal),
        'rms': np.sqrt(np.mean(signal**2)),
        'ptp': np.ptp(signal)
    }


In [9]:
def extract_frequency_features(signal, fs=10):
    freqs, psd = welch(signal, fs=fs, nperseg=256)
    mask = (freqs >= 0.1) & (freqs <= 0.4)
    if np.sum(mask) == 0:
        return {'dominant_freq': np.nan, 'resp_power': np.nan, 'total_power': np.nan}
    
    freqs_band = freqs[mask]
    psd_band = psd[mask]
    
    return {
        'dominant_freq': np.sum(freqs_band*psd_band)/np.sum(psd_band)*60,  # bpm
        'resp_power': np.sum(psd_band),
        'total_power': np.sum(psd)
    }


In [10]:
def extract_edr_features(window, fs=125):
    feats = {}
    
    # Amplitude EDR
    peaks, amps = build_amplitude_edr(window, fs)
    if peaks is not None:
        edr_amp = interpolate_signal(peaks/fs, amps, target_len=window.shape[0])
        edr_amp = downsample_signal(edr_amp, original_fs=fs, target_fs=10)
        edr_amp = bandpass_filter(edr_amp, 0.1, 0.4, fs=10)
        feats.update(extract_time_features(edr_amp))
        feats.update(extract_frequency_features(edr_amp, fs=10))
    else:
        feats.update({k: np.nan for k in ['mean','std','skew','kurtosis','rms','ptp','dominant_freq','resp_power','total_power']})
    
    # RR-Interval EDR
    rr_times, rr_intervals = build_rr_interval_edr(window, fs)
    if rr_times is not None:
        rr_uniform = interpolate_signal(rr_times, rr_intervals, target_len=window.shape[0])
        rr_uniform = downsample_signal(rr_uniform, original_fs=fs, target_fs=10)
        rr_uniform = bandpass_filter(rr_uniform, 0.1, 0.4, fs=10)
        feats_rr = extract_time_features(rr_uniform)
        feats_rr.update({f'rr_{k}': v for k,v in extract_frequency_features(rr_uniform, fs=10).items()})
        feats.update(feats_rr)
    else:
        feats.update({f'rr_{k}': np.nan for k in ['mean','std','skew','kurtosis','rms','ptp','dominant_freq','resp_power','total_power']})
    
    return feats


In [11]:
feature_list = []

for i, window in enumerate(X):
    feats = extract_edr_features(window, fs=125)
    feature_list.append(feats)
    if i % 100 == 0:
        print(f"Processed {i}/{len(X)} windows")

features_df = pd.DataFrame(feature_list)
print(features_df.shape)
features_df.head()


Processed 0/1638 windows
Processed 100/1638 windows
Processed 200/1638 windows
Processed 300/1638 windows
Processed 400/1638 windows
Processed 500/1638 windows
Processed 600/1638 windows
Processed 700/1638 windows
Processed 800/1638 windows
Processed 900/1638 windows
Processed 1000/1638 windows
Processed 1100/1638 windows
Processed 1200/1638 windows
Processed 1300/1638 windows
Processed 1400/1638 windows
Processed 1500/1638 windows
Processed 1600/1638 windows
(1638, 12)


,mean,std,skew,kurtosis,rms,ptp,dominant_freq,resp_power,total_power,rr_dominant_freq,rr_resp_power,rr_total_power
0,-0.000089,0.005012,0.232912,-0.641657,0.005013,0.022100,16.281353,0.003186,0.003311,13.676228,0.000574,0.000594
1,-0.002647,0.019717,-1.241225,3.752492,0.019894,0.126251,17.398293,0.002852,0.003758,16.319873,0.000755,0.001207
2,-0.000309,0.016524,0.556773,5.411445,0.016527,0.122869,18.173261,0.019921,0.020828,17.198748,0.017672,0.018392
3,0.000412,0.005961,0.168945,-0.605352,0.005976,0.026902,16.635780,0.001028,0.001355,19.487258,0.000699,0.000739
4,-0.000665,0.007063,-0.133148,-0.799151,0.007094,0.028969,16.287375,0.002747,0.002827,18.095449,0.000590,0.000623


In [16]:
import pickle
features_df["RR"] = y
with open(r"C:\Users\admin\OneDrive\Desktop\healthcare_ppg\data\processed\features_rr.pkl", "wb") as f:
    pickle.dump(features_df, f)

print("Features saved to Pickle successfully!")


Features saved to Pickle successfully!


In [15]:
features_df["RR"] = y
features_df.to_csv(r"C:\Users\admin\OneDrive\Desktop\healthcare_ppg\data\processed\features_rr.csv", index=False)
print("Re-saved with RR column.")

Re-saved with RR column.
